[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/40_mlops/03_monitoring_and_drift.ipynb)

# Bài 03 — Giám sát mô hình và trôi dữ liệu

**🎯 Mục tiêu**
- Biết **giám sát gì**: dịch vụ (độ trễ, lỗi), dữ liệu đầu vào, dự đoán, hiệu năng (khi nhãn về), chỉ số kinh doanh.
- Phân biệt **trôi dữ liệu** (data/covariate drift — $P(X)$ đổi) và **trôi khái niệm** (concept drift — $P(y\mid X)$ đổi); hiểu vì sao mỗi loại cần cách phát hiện khác nhau.
- Tự cài **PSI**, **KS**, kiểm định chi-bình-phương, **bộ phân loại miền** (domain classifier); hiểu vì sao p-value **không** phải thước đo trôi tốt khi dữ liệu lớn.
- Mô phỏng 12 tuần production: cảnh báo, **độ trễ của nhãn**, huấn luyện lại trên cửa sổ gần đây; xuất metric theo chuẩn Prometheus.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/40_mlops

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from ml_course.datasets import make_customer_churn

FEATURES = ["tenure_months", "contract", "internet_service", "monthly_charges", "total_charges",
            "payment_method", "support_calls", "senior", "n_products"]
NUMERIC = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
CATEGORICAL = ["contract", "internet_service", "payment_method", "senior"]

## 1. Giám sát gì?

Một mô hình **không báo lỗi** khi nó sai dần: API vẫn trả 200, xác suất vẫn nằm trong [0, 1]. Cần giám sát nhiều tầng:

| Tầng | Câu hỏi | Ví dụ metric | Có ngay? |
|---|---|---|---|
| Dịch vụ | API có sống, nhanh, ít lỗi? | số request, tỉ lệ lỗi 5xx/4xx, độ trễ p50/p99, CPU/RAM | ✅ tức thì |
| Dữ liệu đầu vào | Đầu vào có hợp lệ, phân phối có đổi? | tỉ lệ thiếu, giá trị ngoài schema, **PSI**/KS theo đặc trưng | ✅ tức thì |
| Dự đoán | Phân phối đầu ra có đổi? | trung bình xác suất, PSI của điểm, tỉ lệ dự đoán dương | ✅ tức thì |
| Hiệu năng | Mô hình còn đúng không? | AUC, Brier, calibration — **cần nhãn thật** | ⏳ trễ (churn: 30 ngày) |
| Kinh doanh | Mô hình có còn tạo giá trị? | tỉ lệ giữ chân, doanh thu/chiến dịch | ⏳ trễ, nhiễu |

Hai loại trôi:
- **Trôi dữ liệu** (covariate shift): $P(X)$ đổi, quan hệ $P(y\mid X)$ giữ nguyên — vd tăng giá cước, đổi cơ cấu khách hàng. Phát hiện được **ngay** từ đầu vào; mô hình **có thể vẫn tốt**.
- **Trôi khái niệm** (concept drift): $P(y\mid X)$ đổi — vd đối thủ tung khuyến mãi, chính sách công ty thay đổi. Đầu vào có thể **không đổi chút nào** → chỉ phát hiện được qua **nhãn** (trễ).

## 2. Mô hình và dòng dữ liệu 12 tuần

Mô hình huấn luyện trên dữ liệu tham chiếu (reference). Mô phỏng production 12 tuần, mỗi tuần 2.000 khách:
- **Tuần 1–4**: ổn định.
- **Tuần 5–8**: **trôi dữ liệu** — giá cước tăng dần 5%/tuần, tỉ lệ khách hợp đồng tháng tăng. Quan hệ churn giữ nguyên.
- **Tuần 9–12**: thêm **trôi khái niệm** — công ty bắt đầu tặng ưu đãi giữ chân cho mọi khách gọi tổng đài, nên gọi hỗ trợ nhiều giờ lại **ít** rời bỏ (hệ số `support_calls` đổi từ +0.38 thành −0.3). Đầu vào không đổi so với tuần 8.

In [ ]:
def churn_logit(df, calls_effect=0.38):
    """Logit thật của make_customer_churn, có thể đổi hệ số support_calls."""
    return (-1.6 + 1.3 * (df.contract == "month-to-month") - 0.5 * (df.contract == "two_year") - 0.035 * df.tenure_months
            + 0.018 * (df.monthly_charges - 70) + calls_effect * df.support_calls + 0.45 * (df.internet_service == "fiber")
            + 0.35 * (df.payment_method == "e-wallet") + 0.3 * df.senior - 0.2 * (df.n_products - 1))


def production_week(week, n=2000):
    pool = make_customer_churn(4 * n, seed=1000 + week).drop(columns="churn")
    rng = np.random.default_rng(5000 + week)                            # seed KHÁC seed của bộ sinh dữ liệu (tránh tương quan ẩn)
    if week <= 4:
        df = pool.iloc[:n].reset_index(drop=True)
    else:
        k = min(week - 4, 4)
        weights = np.where(pool.contract == "month-to-month", 1 + 0.25 * k, 1.0)
        df = pool.iloc[rng.choice(len(pool), n, replace=False, p=weights / weights.sum())].reset_index(drop=True)
        df["monthly_charges"] = (df.monthly_charges * (1 + 0.05 * k)).round(2)
        df["total_charges"] = (df.total_charges * (1 + 0.05 * k)).round(2)
    calls_effect = 0.38 if week <= 8 else -0.3
    df["churn"] = (rng.random(n) < 1 / (1 + np.exp(-churn_logit(df, calls_effect)))).astype(int)
    return df.assign(week=week)


reference = make_customer_churn(8000, seed=0)
model = make_pipeline(ColumnTransformer([("num", StandardScaler(), NUMERIC), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
                      LogisticRegression(max_iter=1000)).fit(reference[FEATURES], reference.churn)
reference_pred = model.predict_proba(reference[FEATURES])[:, 1]
weeks = {w: production_week(w) for w in range(1, 13)}
print(f"reference: {len(reference):,} khách, churn {reference.churn.mean():.1%} | production: 12 tuần × 2,000 khách")

## 3. Đo trôi: PSI, KS, chi-bình-phương

**PSI** (Population Stability Index) — dùng rộng rãi trong ngành tài chính: chia biến thành các bin theo **phân vị của reference**, so tỉ lệ $r_i$ (reference) và $c_i$ (hiện tại):

$$\text{PSI} = \sum_i (c_i - r_i)\ln\frac{c_i}{r_i}$$

(Đây là KL divergence đối xứng $\text{KL}(c\|r) + \text{KL}(r\|c)$.) Quy ước thường dùng: < 0.1 ổn định · 0.1–0.25 thay đổi vừa · > 0.25 thay đổi lớn.

In [ ]:
def psi(reference, current, bins=10, eps=1e-4):
    """PSI cho biến số: bin theo phân vị của reference."""
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(reference, edges)[0] / len(reference)
    c = np.histogram(current, edges)[0] / len(current)
    r, c = np.clip(r, eps, None), np.clip(c, eps, None)                # tránh log(0) khi bin rỗng
    return float(np.sum((c - r) * np.log(c / r)))


def psi_categorical(reference, current, eps=1e-4):
    categories = sorted(set(reference) | set(current), key=str)          # danh mục mới xuất hiện cũng được tính
    r = np.array([np.mean(reference == k) for k in categories])
    c = np.array([np.mean(current == k) for k in categories])
    r, c = np.clip(r, eps, None), np.clip(c, eps, None)
    return float(np.sum((c - r) * np.log(c / r)))


def ks_statistic(a, b):
    """Khoảng cách lớn nhất giữa hai hàm phân phối tích luỹ thực nghiệm."""
    grid = np.sort(np.concatenate([a, b]))
    cdf_a = np.searchsorted(np.sort(a), grid, side="right") / len(a)
    cdf_b = np.searchsorted(np.sort(b), grid, side="right") / len(b)
    return float(np.max(np.abs(cdf_a - cdf_b)))


w8 = weeks[8]
print(f"monthly_charges tuần 8: PSI = {psi(reference.monthly_charges, w8.monthly_charges):.3f} | "
      f"KS tự viết = {ks_statistic(reference.monthly_charges.values, w8.monthly_charges.values):.4f} | scipy = {stats.ks_2samp(reference.monthly_charges, w8.monthly_charges).statistic:.4f}")
table = pd.crosstab(np.r_[["ref"] * len(reference), ["w8"] * len(w8)], np.r_[reference.contract, w8.contract])
chi2, p_chi2, *_ = stats.chi2_contingency(table)
print(f"contract tuần 8: PSI = {psi_categorical(reference.contract.values, w8.contract.values):.3f} | chi-bình-phương = {chi2:.1f}, p = {p_chi2:.1e}")

**Vì sao không dùng p-value làm thước đo trôi?** Với dữ liệu lớn, kiểm định phát hiện cả những thay đổi **nhỏ không đáng kể**: thử tăng cước đúng **1%** (không ảnh hưởng gì tới mô hình) với các cỡ mẫu khác nhau.

In [ ]:
rng = np.random.default_rng(0)
base_charges = make_customer_churn(200_000, seed=7).monthly_charges.values
for n in [500, 5_000, 50_000]:
    ref_sample, cur_sample = rng.choice(base_charges, n), rng.choice(base_charges, n) * 1.01
    print(f"n = {n:6,}: KS p-value = {stats.ks_2samp(ref_sample, cur_sample).pvalue:.1e} | PSI = {psi(ref_sample, cur_sample):.4f}")

P-value đo "có khác **không**", phụ thuộc mạnh vào cỡ mẫu — giám sát 50 đặc trưng mỗi ngày với hàng chục nghìn dòng thì **luôn** có báo động. PSI (và KS statistic, Wasserstein, Jensen–Shannon) đo "khác **bao nhiêu**" — ổn định theo cỡ mẫu → đặt ngưỡng theo **độ lớn**. (PSI với mẫu nhỏ có chệch dương do nhiễu lấy mẫu — cần đủ dữ liệu mỗi cửa sổ.)

## 4. Theo dõi 12 tuần

Mỗi tuần tính: PSI các đặc trưng, PSI và trung bình của dự đoán (có ngay); AUC, Brier, tỉ lệ churn thật (chỉ biết sau khi nhãn về — 30 ngày ≈ 4 tuần sau).

In [ ]:
rows = []
for w, df in weeks.items():
    pred = model.predict_proba(df[FEATURES])[:, 1]
    row = {"tuần": w}
    for col in NUMERIC:
        row[f"PSI {col}"] = psi(reference[col], df[col])
    for col in CATEGORICAL:
        row[f"PSI {col}"] = psi_categorical(reference[col].values, df[col].values)
    row |= {"PSI dự đoán": psi(reference_pred, pred), "dự đoán TB": pred.mean(), "churn thật": df.churn.mean(),
            "AUC": roc_auc_score(df.churn, pred), "Brier": brier_score_loss(df.churn, pred)}
    rows.append(row)
monitor = pd.DataFrame(rows).set_index("tuần")
feature_psi = monitor[[c for c in monitor if c.startswith("PSI ") and c != "PSI dự đoán"]]
monitor["PSI lớn nhất"] = feature_psi.max(axis=1)
monitor["đặc trưng trôi nhất"] = feature_psi.idxmax(axis=1).str.replace("PSI ", "")
print(monitor[["PSI lớn nhất", "đặc trưng trôi nhất", "PSI dự đoán", "dự đoán TB", "churn thật", "AUC", "Brier"]].round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for col in ["monthly_charges", "contract", "support_calls"]:
    axes[0].plot(monitor.index, monitor[f"PSI {col}"], marker="o", label=col)
axes[0].axhline(0.25, color="red", ls="--", lw=1, label="ngưỡng 0.25")
axes[0].set_title("Trôi đầu vào (PSI)")
axes[1].plot(monitor.index, monitor["dự đoán TB"], marker="o", label="xác suất dự đoán TB")
axes[1].plot(monitor.index, monitor["churn thật"], marker="s", label="tỉ lệ churn thật")
axes[1].set_title("Dự đoán vs thực tế")
axes[2].plot(monitor.index, monitor["AUC"], marker="o", color="C3")
axes[2].set_title("ROC-AUC (cần nhãn)")
for ax in axes:
    ax.axvspan(4.5, 8.5, color="orange", alpha=0.08)
    ax.axvspan(8.5, 12.5, color="red", alpha=0.08)
    ax.set_xlabel("tuần")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

Đọc bảng và biểu đồ (vùng cam: trôi dữ liệu; vùng đỏ: thêm trôi khái niệm):
- **Tuần 5–8**: PSI của `monthly_charges` vượt xa ngưỡng — báo động đỏ. Nhưng AUC giữ nguyên và xác suất dự đoán trung bình **bám sát** tỉ lệ churn thật: mô hình (đúng dạng logistic) **ngoại suy tốt** khi chỉ $P(X)$ đổi. Báo động trôi dữ liệu ≠ mô hình hỏng — cần điều tra, không tự động huấn luyện lại.
- **Tuần 9–12**: PSI đầu vào và PSI dự đoán giữ **nguyên mức của tuần 8** — không có tín hiệu **mới** nào cho biết điều gì đã đổi. Nhưng tỉ lệ churn thật **giảm mạnh** trong khi mô hình vẫn dự đoán như cũ, AUC tụt rõ. Chỉ phát hiện được khi **nhãn về** — với churn là ~4 tuần sau.

## 5. Bộ phân loại miền (domain classifier)

PSI xét **từng** đặc trưng riêng lẻ; trôi có thể nằm ở **tương quan** giữa các đặc trưng. Cách tổng quát: huấn luyện một bộ phân loại phân biệt "reference" và "hiện tại". AUC ≈ 0.5 → không phân biệt được (không trôi); AUC cao → có trôi, và độ quan trọng đặc trưng chỉ ra trôi ở đâu.

In [ ]:
def domain_classifier(reference_df, current_df, seed=0):
    X = pd.concat([reference_df[FEATURES], current_df[FEATURES]], ignore_index=True)
    X[CATEGORICAL] = OrdinalEncoder().fit_transform(X[CATEGORICAL].astype(str))
    y = np.r_[np.zeros(len(reference_df)), np.ones(len(current_df))]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = HistGradientBoostingClassifier(max_iter=100, random_state=seed).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
    importance = permutation_importance(clf, X_te, y_te, scoring="roc_auc", n_repeats=3, random_state=seed)
    return auc, pd.Series(importance.importances_mean, index=FEATURES).sort_values(ascending=False)


sample_ref = reference.sample(2000, random_state=0)
for w in [2, 8]:
    auc, importance = domain_classifier(sample_ref, weeks[w])
    print(f"tuần {w}: AUC bộ phân loại miền = {auc:.3f} | đặc trưng phân biệt nhất: {importance.head(3).round(3).to_dict()}")

Tuần 2: AUC ≈ 0.5 — hai tập không phân biệt được. Tuần 8: AUC cao; đặc trưng phân biệt nhất là **cước** — và cả `internet_service`, dù tỉ lệ các loại dịch vụ gần như không đổi. Bộ phân loại dùng **kết hợp** "cước so với loại gói" (cùng một gói giờ đắt hơn): trôi nằm trong **quan hệ giữa các đặc trưng**, thứ PSI từng biến không thấy. Bộ phân loại miền cũng không thấy trôi khái niệm (nó chỉ nhìn $X$).

## 6. Độ trễ của nhãn và cảnh báo

Quy tắc cảnh báo ví dụ — mỗi tuần, đánh giá những gì **đã biết tại thời điểm đó** (nhãn trễ 4 tuần):
- **Trôi dữ liệu**: PSI lớn nhất > 0.25 → mở ticket điều tra.
- **Trôi dự đoán**: PSI dự đoán > 0.1.
- **Hiệu năng** (của tuần $t - 4$, nhãn vừa về): AUC giảm > 0.05 so với mức triển khai, hoặc |dự đoán TB − churn thật| > 0.05.

In [ ]:
LABEL_DELAY = 4
baseline_auc = monitor.loc[1:4, "AUC"].mean()
alerts = []
for w in range(1, 17):                                                 # tiếp tục theo dõi 4 tuần sau khi dữ liệu tuần 12 đến
    known = w - LABEL_DELAY                                            # tuần có nhãn mới về
    alert = {"tuần": w}
    if w <= 12:
        alert["trôi dữ liệu"] = "⚠️" if monitor.loc[w, "PSI lớn nhất"] > 0.25 else ""
        alert["trôi dự đoán"] = "⚠️" if monitor.loc[w, "PSI dự đoán"] > 0.1 else ""
    if 1 <= known <= 12:
        gap = abs(monitor.loc[known, "dự đoán TB"] - monitor.loc[known, "churn thật"])
        bad = baseline_auc - monitor.loc[known, "AUC"] > 0.05 or gap > 0.05
        alert["hiệu năng (nhãn của tuần)"] = f"tuần {known}: " + ("🚨" if bad else "ok")
    alerts.append(alert)
print(pd.DataFrame(alerts).fillna("").set_index("tuần").to_string())

Cảnh báo trôi dữ liệu sáng từ tuần 6 (đúng), còn cảnh báo trôi dự đoán **chập chờn** quanh ngưỡng từ tuần 9 — hệ quả của trôi dữ liệu đã có, không phải dấu hiệu của trôi khái niệm (ngưỡng sát mức nhiễu gây "mệt mỏi vì cảnh báo"). Trôi khái niệm bắt đầu ở tuần 9 nhưng cảnh báo hiệu năng đầu tiên chỉ đến ở **tuần 13** — 4 tuần mô hình chạy sai mà không ai biết. Cách rút ngắn:
- **Nhãn sớm/đại diện** (proxy): vd khách không đăng nhập 7 ngày — có sau 1 tuần thay vì 30 ngày.
- **Nhóm đối chứng** (holdout traffic): một phần nhỏ khách không bị chiến dịch tác động để đo hiệu năng không chệch.
- **Theo dõi sự kiện kinh doanh**: chiến dịch, đổi giá, đổi chính sách — trôi khái niệm thường có **nguyên nhân biết trước**. Ở đây chính hành động "tặng ưu đãi cho người gọi tổng đài" đã làm đổi quan hệ — một dạng **vòng phản hồi**.

## 7. Phản ứng: huấn luyện lại

Khi nhãn tuần 9–10 đã về, huấn luyện lại và đánh giá trên tuần 11–12 (dữ liệu chưa thấy):

In [ ]:
def fit_churn_model(df):
    return make_pipeline(ColumnTransformer([("num", StandardScaler(), NUMERIC), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
                         LogisticRegression(max_iter=1000)).fit(df[FEATURES], df.churn)


recent = pd.concat([weeks[9], weeks[10]], ignore_index=True)
evaluation = pd.concat([weeks[11], weeks[12]], ignore_index=True)
candidates = {
    "mô hình cũ (reference)": model,
    "huấn luyện lại: reference + mọi tuần": fit_churn_model(pd.concat([reference] + [weeks[w] for w in range(1, 11)], ignore_index=True)),
    "huấn luyện lại: chỉ tuần 9–10": fit_churn_model(recent),
}
for name, m in candidates.items():
    p = m.predict_proba(evaluation[FEATURES])[:, 1]
    print(f"{name:38s}: AUC = {roc_auc_score(evaluation.churn, p):.3f} | Brier = {brier_score_loss(evaluation.churn, p):.3f} | dự đoán TB = {p.mean():.3f} (thật {evaluation.churn.mean():.3f})")

- Gộp **toàn bộ** lịch sử: dữ liệu cũ (quan hệ cũ) chiếm đa số, kéo mô hình về khái niệm cũ → cải thiện ít.
- Cửa sổ **gần đây**: học đúng quan hệ mới, dù ít dữ liệu hơn. Đánh đổi: cửa sổ ngắn thích nghi nhanh nhưng nhiễu; cửa sổ dài ổn định nhưng chậm. Lựa chọn khác: **trọng số giảm dần theo thời gian**.
- Chiến lược huấn luyện lại: **định kỳ** (đơn giản, dễ dự đoán chi phí) hoặc **theo kích hoạt** (khi cảnh báo hiệu năng) — mô hình mới luôn qua cổng thăng hạng (chương 39) trước khi thay thế.

## 8. Xuất metric cho Prometheus

Trong production, dịch vụ **xuất** metric qua endpoint `/metrics`; Prometheus định kỳ thu thập, Grafana vẽ dashboard, Alertmanager gửi cảnh báo. Thư viện `prometheus_client` (cài kèm `[mlops]`):

In [ ]:
try:
    from prometheus_client import CollectorRegistry, Counter, Gauge, Histogram, generate_latest
except ImportError:
    CollectorRegistry = None
    print('Chưa cài prometheus_client — bỏ qua: pip install prometheus-client')

if CollectorRegistry is not None:
    registry = CollectorRegistry()
    requests_total = Counter("churn_requests_total", "Số request dự đoán", ["status"], registry=registry)
    latency = Histogram("churn_latency_seconds", "Độ trễ dự đoán", buckets=(0.005, 0.01, 0.025, 0.05, 0.1), registry=registry)
    feature_psi_gauge = Gauge("churn_feature_psi", "PSI theo đặc trưng (so với reference)", ["feature"], registry=registry)
    sim = np.random.default_rng(1)
    for _ in range(500):                                               # mô phỏng 500 request
        requests_total.labels(status="200" if sim.random() > 0.01 else "422").inc()
        latency.observe(sim.lognormal(np.log(0.008), 0.4))
    for col in ["monthly_charges", "contract", "support_calls"]:
        feature_psi_gauge.labels(feature=col).set(monitor.loc[8, f"PSI {col}"])
    exposition = generate_latest(registry).decode()
    print("\n".join(line for line in exposition.splitlines() if not line.startswith("#") and "_created" not in line)[:1400])

Prometheus lưu chuỗi thời gian; quy tắc cảnh báo viết bằng PromQL, ví dụ `histogram_quantile(0.99, rate(churn_latency_seconds_bucket[5m])) > 0.05` (p99 > 50 ms) hay `max(churn_feature_psi) > 0.25`. Công cụ chuyên cho giám sát ML: Evidently, NannyML (ước lượng hiệu năng **khi chưa có nhãn**), WhyLabs, Arize, Fiddler.

## 🧠 Kiểm tra nhanh

1. Kể năm tầng giám sát. Tầng nào có tín hiệu ngay, tầng nào trễ?
2. Trôi dữ liệu và trôi khái niệm khác nhau thế nào? Mỗi loại phát hiện bằng cách nào?
3. PSI tính thế nào? Ngưỡng thường dùng?
4. Vì sao p-value của KS không phải thước đo trôi tốt với dữ liệu lớn?
5. Bộ phân loại miền phát hiện điều gì mà PSI từng đặc trưng bỏ sót?
6. Độ trễ của nhãn gây ra vấn đề gì? Ba cách giảm thiểu?
7. Khi huấn luyện lại sau trôi khái niệm, nên dùng toàn bộ lịch sử hay cửa sổ gần đây?

## 📌 Tóm tắt

- Giám sát nhiều tầng: dịch vụ, đầu vào, dự đoán (tức thì) và hiệu năng, kinh doanh (trễ).
- Trôi dữ liệu thấy được ngay nhưng không nhất thiết làm hỏng mô hình; trôi khái niệm có thể vô hình với mọi giám sát đầu vào — cần nhãn, nhãn proxy, nhóm đối chứng.
- Đo trôi bằng độ lớn (PSI, KS statistic, bộ phân loại miền), không bằng p-value.
- Cảnh báo theo những gì đã biết tại thời điểm đó; huấn luyện lại trên cửa sổ phù hợp và luôn qua cổng thăng hạng.

## 📚 Tài liệu tham khảo

- Gama et al. (2014) — *A Survey on Concept Drift Adaptation*
- Rabanser, Günnemann & Lipton (2019) — *Failing Loudly: An Empirical Study of Methods for Detecting Dataset Shift*
- Quiñonero-Candela et al. (2009) — *Dataset Shift in Machine Learning*
- Huyen (2022) — *Designing Machine Learning Systems*, chương 8 (Data Distribution Shifts and Monitoring)
- Google SRE Book — *Monitoring Distributed Systems* (bốn tín hiệu vàng: latency, traffic, errors, saturation)
- Tài liệu Prometheus — prometheus.io/docs